In [2]:
import mysql.connector

In [7]:
pip install mysql-connector-python



   ---------------------------------------- 0.0/16.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.1 MB ? eta -:--:--
   ---------------------------------------- 0.1/16.1 MB 1.7 MB/s eta 0:00:10
    --------------------------------------- 0.2/16.1 MB 3.1 MB/s eta 0:00:06
   - -------------------------------------- 0.7/16.1 MB 5.2 MB/s eta 0:00:03
   -- ------------------------------------- 1.0/16.1 MB 6.3 MB/s eta 0:00:03
   --- ------------------------------------ 1.4/16.1 MB 7.3 MB/s eta 0:00:03
   ---- ----------------------------------- 2.0/16.1 MB 7.9 MB/s eta 0:00:02
   ------ --------------------------------- 2.6/16.1 MB 8.3 MB/s eta 0:00:02
   ------- -------------------------------- 3.2/16.1 MB 9.1 MB/s eta 0:00:02
   --------- ------------------------------ 3.6/16.1 MB 9.0 MB/s eta 0:00:02
   ---------- ----------------------------- 4.1/16.1 MB 9.1 MB/s eta 0:00:02
   ----------- ----


[notice] A new release of pip is available: 24.0 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
conn = mysql.connector.connect(
    host="localhost",  
    user="root",
    password="1234",
    database="warehouse"
)

cursor = conn.cursor()


In [ ]:
API_URL = "http://127.0.0.1:8000/api/barcodes/"  

In [ ]:
# def send_to_backend(data):
#     payload = {
         
#     }

In [4]:
# Import necessary libraries
import cv2  # OpenCV for image processing
import csv  # CSV module for saving barcode data
from pyzbar.pyzbar import decode  # Pyzbar for decoding barcodes

# Barcode Detector Class

# If a barcode is detected, we need to check the database (identification table) to see whether this barcode ID exists.
class BarcodeDetector:

    def decode_barcode(enhanced_roi):
        """Decode barcode from an enhanced region of interest (ROI) and return the most confident result."""
        barcodes = decode(enhanced_roi)
        
        if not barcodes:
            return None
    
        # Select the longest barcode data (heuristic for best confidence)
        best_barcode = max(barcodes, key=lambda b: len(b.data), default=None)
        
        if best_barcode:
            data = best_barcode.data.decode('utf-8')
            barcode_type = best_barcode.type
            
            # check if the barcode exists in the database
            exists = self.update_asset_status(data)
            if exists == True:
                print("found")
                product_info = self.get_product_info(data)
            # Display product details if it exists
            if product_info:
                print(f"Product Found: {product_info}")  
            else:
                print(f"Barcode {data} NOT found in the database.")
            
            return f"{data} ({barcode_type})"
        
        return None

    
    # def decode_barcodes(self, img, roi):
    #     barcodes = decode(roi)  # Decode all barcodes in the region of interest
    #     for barcode in barcodes:
    #         data = barcode.data.decode('utf-8')  # Extract and decode barcode data
    #         barcode_type = barcode.type  # Get the barcode type (e.g., QR Code, EAN-13, etc.)
    #         (x, y, w, h) = barcode.rect  # Get barcode bounding box coordinates

    #         # Draw a green rectangle around the detected barcode
    #         cv2.rectangle(img, (x, y), (x + w, y + h), (0, 255, 0), 2)

    #         # Create a text label with barcode data and type
    #         text = f"{data} ({barcode_type})"
    #         cv2.putText(img, text, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)

    #         # check if the barcode exists in the database
    #         exists = self.update_asset_status(data)
    #         if exists == True:
    #             print("found")
    #             product_info = self.get_product_info(data)
    #         # Display product details if it exists
    #         if product_info:
    #             print(f"Product Found: {product_info}")  
    #         else:
    #             print(f"Barcode {data} NOT found in the database.")

    # This function checks if the information of the product exists. If it does then it will output that info
    def update_asset_status(self,barcode_id):
        query = "SELECT * FROM identification WHERE IdentifierValue = %s"
        execution = cursor.execute(query, (barcode_id,))
        resultset = cursor.fetchall()
        for row in resultset:
            # We need only asset id to get the details of the asset
            asset_id = row[1]  
            print("AssetID:", asset_id)
        if resultset:
            assetFound = True
            query = "UPDATE asset SET AssetFound = 1 WHERE AssetID = %s"
            execution = cursor.execute(query, (asset_id,))            
            conn.commit()
        else: 
            assetFound = False
        return assetFound

    def get_product_info(self, barcode_id):
        # First we select the Asset ID from the table
        query = "SELECT * FROM identification WHERE IdentifierValue = %s"
        execution = cursor.execute(query, (barcode_id,))
        resultset = cursor.fetchall()
        for row in resultset:
            # We need only asset id to get the details of the asset
            asset_id = row[1]  
            print("AssetID:", asset_id)
            
        # This provides the data of the asset 
        query = "SELECT * FROM asset WHERE AssetID = %s"
        execution = cursor.execute(query, (asset_id,))
        resultset = cursor.fetchone()  # Fetch the product info if found
        product_id = resultset[1]
        assetLocation = resultset[2]
        assetCondition = resultset[3]
        dateadded = resultset[4]
        assetDispatched = resultset[6]
        print(assetLocation)

        # Getting additional information from the product tabls
        query = "SELECT * FROM product WHERE ProductID = %s"
        execution = cursor.execute(query, (product_id,))
        resultset = cursor.fetchall()
        for row in resultset:
            productName = row[2]
            productCategory = row[3]
            # productQuantity = row[5]
            productPrice = row[5]
            print(productName)
            print(productPrice)
        # conn.close()  # Close database connection
        return resultset  # Returns product details (e.g., name, price) or None
        
# Initialize video capture (Webcam)
cap = cv2.VideoCapture(0)  # Open the default camera (index 0)

# Set the resolution of the video capture
cap.set(3, 720)  # Set width to 720 pixels
cap.set(4, 720)  # Set height to 720 pixels


True

In [8]:
import cv2
import csv
import math
from ultralytics import YOLO

class QRCodeDetector:

    def decode_qrcode(enhanced_roi):
    """Decode a QR code from an enhanced region of interest (ROI)."""
    decoded_info = decode(enhanced_roi)
    
    # if not decoded_info:
    #     return None  # Return None if no QR code is detected

    # Extract and return the first decoded QR code data
    for obj in decoded_info:
        qr_data = obj.data.decode('utf-8')
        product_info = None  

        exists = self.update_asset_status(qr_data)
        if exists == True:
            print("found")
            product_info = self.get_product_info(qr_data)
        # Display product details if it exists
        if product_info:
            print(f"Product Found: {product_info}")  
        else:
            print(f"QR Code {data} NOT found in the database.")

        return qr_data  # Return the first detected QR code
    
    return None  # Fallback return (should never reach here)

    
    # def decode_qrcodes(self, img, roi):
    #     detector = cv2.QRCodeDetector()
    #     data, bbox, _ = detector.detectAndDecode(roi)


    #     print(f"Decoded Data: {data}")  # Debugging output
    #     if bbox is not None:
    #         print("Bounding box detected for QR code.")
    #     else:
    #         print("No bounding box detected.")
        
    #     product_info = None  

    #     exists = self.update_asset_status(data)
    #     if exists == True:
    #         print("found")
    #         product_info = self.get_product_info(data)
    #     # Display product details if it exists
    #     if product_info:
    #         print(f"Product Found: {product_info}")  
    #     else:
    #         print(f"QR Code {data} NOT found in the database.")

    #     if bbox is not None:
    #         bbox = bbox.astype(int)
    #         for i in range(len(bbox)):
    #             start_point = tuple(bbox[i][0])
    #             end_point = tuple(bbox[(i + 1) % len(bbox)][0])
    #             cv2.line(img, start_point, end_point, color=(255, 0, 0), thickness=2)
    #         cv2.putText(img, data, (int(bbox[0][0][0]), int(bbox[0][0][1]) - 10), 
    #                     cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 250, 120), 2)

    # This function checks if the information of the product exists. If it does then it will output that info
    def update_asset_status(self,barcode_id):
        query = "SELECT * FROM identification WHERE IdentifierValue = %s"
        execution = cursor.execute(query, (barcode_id,))
        resultset = cursor.fetchall()
        for row in resultset:
            # We need only asset id to get the details of the asset
            asset_id = row[1]  
            print("AssetID:", asset_id)
        if resultset:
            assetFound = True
            query = "UPDATE asset SET AssetFound = 1 WHERE AssetID = %s"
            execution = cursor.execute(query, (asset_id,))            
            conn.commit()
        else: 
            assetFound = False
        return assetFound

    def get_product_info(self, barcode_id):
        # First we select the Asset ID from the table
        query = "SELECT * FROM identification WHERE IdentifierValue = %s"
        execution = cursor.execute(query, (barcode_id,))
        resultset = cursor.fetchall()
        for row in resultset:
            # We need only asset id to get the details of the asset
            asset_id = row[1]  
            print("AssetID:", asset_id)
            
        # This provides the data of the asset 
        query = "SELECT * FROM asset WHERE AssetID = %s"
        execution = cursor.execute(query, (asset_id,))
        resultset = cursor.fetchone()  # Fetch the product info if found
        product_id = resultset[1]
        assetLocation = resultset[2]
        assetCondition = resultset[3]
        dateadded = resultset[4]
        assetDispatched = resultset[6]
        print(assetLocation)

        # Getting additional information from the product tabls
        query = "SELECT * FROM product WHERE ProductID = %s"
        execution = cursor.execute(query, (product_id,))
        resultset = cursor.fetchall()
        for row in resultset:
            productName = row[2]
            productCategory = row[3]
            # productQuantity = row[5]
            productPrice = row[5]
            print(productName)
            # print(productPrice)
        # conn.close()  # Close database connection
        return resultset  # Returns product details (e.g., name, price) or None

In [10]:
import csv
# def initialize_csv():
#     # with open("Database.csv", mode='w') as csvfile:
    #     csvfileWriter = csv.writer(csvfile, delimiter=',', quotechar='"', quoting=csv.QUOTE_ALL)
    #     csvfileWriter.writerow(["Data", "Detection Count"])

def enhance_roi(roi):
    """Enhance the region of interest (ROI) for better barcode detection."""
    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)

    # Increase contrast and brightness (alpha=contrast, beta=brightness)
    enhanced = cv2.convertScaleAbs(gray, alpha=1.5, beta=15)  # Adjust values if needed

    # Apply adaptive thresholding
    enhanced = cv2.adaptiveThreshold(enhanced, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                     cv2.THRESH_BINARY, 11, 2)

    # Apply morphological operations to reduce noise
    #kernel = np.ones((1, 1), np.uint8)
    #enhanced = cv2.morphologyEx(enhanced, cv2.MORPH_CLOSE, kernel)

    return enhanced



def main():

    # Initialize the BarcodeDetector and QRCodeDetector
    barcode_detector_manual = BarcodeDetector()
    qrcode_detector_manual = QRCodeDetector()
        
    # Initialize OpenCV's barcode and QR code detectors
    barcode_detector = cv2.barcode_BarcodeDetector()
    qrcode_detector = cv2.QRCodeDetector()
    
    # Create a window for display
    cv2.namedWindow("FrameOut", cv2.WINDOW_NORMAL)
    
    # Open webcam
    cap = cv2.VideoCapture(1)
    if not cap.isOpened():
        print("Error: Could not open video capture device.")
        exit()
    
    while True:
        ret, frame = cap.read()
        if not ret:
            print("Failed to capture frame. Exiting.")
            break
    
        # Barcode Detection and Decoding
        ret_val, decoded_info, decoded_type, corners = barcode_detector.detectAndDecodeWithType(frame)
        if corners is not None and len(corners) > 0:
            cv2.putText(frame, "Barcode Detected", (50, 50), cv2.FONT_HERSHEY_COMPLEX, 1, (0, 255, 255), 2)
            
            for i in range(len(decoded_info)):
                pts = corners[i].reshape((-1, 1, 2)).astype(np.int32)
                cv2.polylines(frame, [pts], isClosed=True, color=(0, 255, 255), thickness=2)
                
                # Extract and enhance ROI
                x, y, w, h = cv2.boundingRect(pts)
                padding = 5
                x, y = max(0, x - padding), max(0, y - padding)
                w, h = min(frame.shape[1] - x, w + 2 * padding), min(frame.shape[0] - y, h + 2 * padding)
                roi = frame[y:y+h, x:x+w].copy()
                
                if roi.size == 0:
                    continue
                
                enhanced_roi = enhance_roi(roi)
                
                # Decode barcode
                start_time = time.time()
                decoded_value = barcode_detector_manual.decode_barcode(enhanced_roi)
                end_time = time.time()
                decode_time = end_time - start_time
                
                if roi.shape[0] > 0 and roi.shape[1] > 0:
                    cv2.imshow("ROI", enhanced_roi)
                    if decoded_value is not None:
                        print(f"Decoded from enhanced ROI_barcode: {decoded_value}")
        
        # QR Code Detection and Decoding
        qr_codes = decode(frame)
        #print("Original Decoded QR Code Data:", qr_codes)
    
        for qr in qr_codes:
            points = qr.polygon
            if len(points) == 4:
                points = np.array([(point.x, point.y) for point in points], dtype=np.int32).reshape((-1, 1, 2))
                cv2.polylines(frame, [points], isClosed=True, color=(0, 0, 255), thickness=2)
                
                x, y, w, h = cv2.boundingRect(points)
                roi = frame[y:y+h, x:x+w]
                
                enhanced_roi = cv2.convertScaleAbs(roi, alpha=1.5, beta=50)
                qr_data = qrcode_detector_manual.decode_qrcode(enhanced_roi)
                
                if qr_data is not None:
                    print("Decoded QR Code Data:", qr_data)            
                
                cv2.imshow("Enhanced ROI", enhanced_roi)
        
        # Display output frame
        cv2.imshow("FrameOut", frame)
        
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
    
    cap.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    main()

Choose Detection Mode:
1. QR Code Detection
2. Barcode Detection

0: 480x640 (no detections), 92.2ms
Speed: 7.8ms preprocess, 92.2ms inference, 0.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 114.5ms
Speed: 4.4ms preprocess, 114.5ms inference, 0.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 114.0ms
Speed: 0.0ms preprocess, 114.0ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 98.9ms
Speed: 0.0ms preprocess, 98.9ms inference, 0.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 99.0ms
Speed: 0.0ms preprocess, 99.0ms inference, 0.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 110.3ms
Speed: 1.4ms preprocess, 110.3ms inference, 0.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 96.0ms
Speed: 0.0ms preprocess, 96.0ms inference, 0.0ms postprocess per image at shape (1, 3, 480, 640